# 34 — Memory v2 Validation Run (10 agents × 4 days, local Qwen3)

End-to-end validation of the v2 tiered-memory refactor on a real local-Qwen3 run via `mlx_lm.server`. Cribbed structurally from NB 33 (sandbox structural checks) but here every section is checked against artefacts produced by an actual `run_simulation` call rather than a mocked agent. Provider/model match the research canon used in NB 29 / NB 32 and the eventual HPC/AIRE runs (same `provider='local'` path, identical `cag.io.llm` plumbing).

> **Post-run design change.** This notebook recorded the only at-scale execution of the `compress_day0_anchor` step (60 LLM calls compressing ~85-word Day-0 rationales into ~52-word anchors — a ~39% reduction). Analysis afterwards showed (a) the compression ratio was modest, (b) the `_section_day0_anchor` fallback already surfaces the verbatim Day-0 rationale, and (c) one-shot LLM rephrasing of a permanent reference point added error risk. The compression step was therefore **removed** from `agent.py` / `sim.py` / `results.py` / `checkpoint.py` and the §6 cells were deleted from this notebook. The instrumentation cell still tracks an `anchor_compression` bucket but it will now stay at `0`. The §6 scoreboard entry was also removed. The net v2-vs-v1 LLM cost delta is now **zero extra calls**.

**What the v2 refactor changed**
- ~~Identity anchor compressed end of Day 0 (`compress_day0_anchor` → `day0_anchors.csv`).~~ *(removed — see banner above)*
- `manage_memory` (day d-2 → daily summary) now runs BEFORE the EOD survey so day d's survey can see it.
- EOD survey policy order shuffled deterministically per day (`random.Random(seed*1000 + day)`) — kills the v1 fixed-1→6 position bias.
- `assemble_context` rewritten with target-scoped anchor / recent-own-reasoning / today-so-far sections. The Day-0 anchor section now reads the verbatim Day-0 rationale directly from `survey_reasoning`.
- Unified `compress_daily_memory` pulls reflections + every same-day `survey_reasoning` row across all policies in package mode.

**What this notebook does**
1. Ping the local `mlx_lm.server` (`http://localhost:8080/v1`) — fail-fast if unreachable, before any agents are built.
2. Build a 10-citizen sample nation from YouGov (same construction as NB 17/19/29/32).
3. Wrap `cag.abm.agent.send_chat` with a categorising counter (daily-summary / other).
4. Run `run_simulation` for 4 alternating package-mode days × all 6 policies × local Qwen3 × `debias=True`, timed.
5. Report wall-clock + per-category LLM call breakdown.
6. Validate (with pass/fail):
   - **§7** `manage_memory` before EOD — `Summary of recent days:` block appears in EOD contexts on day 3+ but not earlier; `Day {d-2}:` marker present.
   - **§8** EOD shuffle — same across agents within a day, different across days, matches deterministic `random.Random` reproduction exactly.
   - **§9** Today-so-far block — absent in 1st-of-day survey, present in later-of-day surveys.
   - **§10** Full assembled-context dump + headers-in-order check.
   - **§11** Unified daily-summary — row count = `n_agents × max(0, n_days - 2)`, all `PACKAGE_SCOPE`-tagged.
   - **§12** Scoreboard.

**Defaults used (all canonical research defaults except size)**
- `communication_mode='package'` over all 6 climate policies
- `day0_anchor='ground_truth_with_rationale'` (so each agent has a Day-0 rationale in `survey_reasoning` for the new verbatim-anchor section to surface)
- `debias=True`, `thinking=True`, `llm_temperature=0.5`
- `political_message_source='offline'`, set `v1`
- Default exposure (`rule_affinity_rank`, balanced weights, committed-minority symmetric targets)
- Symmetric reach (`reach_a=reach_b=1.0`), no audience cap, `k_peers_per_day=2`
- Default stochastic-block network (`p_intra=0.15`, `p_inter=0.05`)
- `random_seed=42`
- LLM: `provider='local'`, `model='mlx-community/Qwen3-8B-4bit'`, served by `mlx_lm.server` on `http://localhost:8080/v1` — same path the HPC/AIRE runs will use.

**Days plan.** 4 alternating package-mode days, P-A / P-B order flipped each day so neither side has a systematic first-mover advantage in the within-day broadcast:
- Day 1: `[P-A, P-B, C]`
- Day 2: `[P-B, P-A, C]`
- Day 3: `[P-A, P-B, C]`
- Day 4: `[P-B, P-A, C]`

**Cost / wall-time.** No API spend (local). Expect tens of minutes wall-clock at this size on a Mac with `mlx_lm.server` — Qwen3 is slower per call than gpt-5-mini and `thinking=True` adds further latency. Adjust expectations accordingly when comparing to the previous gpt-5-mini run.


## 1. Imports + local-server ping

Fail-fast if the local `mlx_lm.server` is unreachable, before we build any agents.


In [ ]:
import os, sys, random, logging, time, json
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cag.io.llm import ping_local, configure_local
from cag.io.survey import load
from cag.abm.political_messages import load_message_pool, PACKAGE_KEY
from cag.abm.attributes.opinion import ALL_CLIMATE_POLICIES, ClimatePolicyID

LOCAL_BASE_URL = 'http://localhost:8080/v1'
LOCAL_MODEL    = 'mlx-community/Qwen3-8B-4bit'

info = ping_local(base_url=LOCAL_BASE_URL)
print('Local server reachable. /v1/models returned:', info)
print(f'Provider/model : local/{LOCAL_MODEL}')
print(f'Base URL       : {LOCAL_BASE_URL}')


## 2. Build a 10-agent SurveyedNation

Same construction as NB17/NB19/NB29, just smaller.

In [ ]:
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

N_CITIZENS  = 10
RANDOM_SEED = 42
year = 2026
random.seed(RANDOM_SEED)

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place='UK',
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load('../data/yougov_survey_data/YouGovProcessedData.csv')
data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

print(f'Citizens loaded: {len(sn.agents_active)} (target {N_CITIZENS})')

## 3. Canonical SIM_CONFIG (every key explicit, override only size + provider)

Every key in `cag.abm.sim.SIM_CONFIG` is listed below.

- `# default` — value equals the source default.
- `# SMOKE OVERRIDE` — size knob (`n_citizens`, `days`).
- `# PROVIDER OVERRIDE` — switched from local Qwen3 to OpenAI `gpt-5-mini` per request.

Everything else matches the v0.6 research-canon defaults.

In [ ]:
from cag.abm.sim import SIM_CONFIG

config = {
    # ---- Population --------------------------------------------------------
    'n_citizens'                 : N_CITIZENS,                          # SMOKE OVERRIDE (full=100)

    # ---- Day plan ----------------------------------------------------------
    # 4 alternating package-mode days. P-A/P-B order flipped each day so
    # neither side has a systematic first-mover advantage in the within-day
    # broadcast. Closing phase C (peer messaging) is always last.
    'days': [                                                           # SMOKE OVERRIDE
        {'phases': ['P-A', 'P-B', 'C']},
        {'phases': ['P-B', 'P-A', 'C']},
        {'phases': ['P-A', 'P-B', 'C']},
        {'phases': ['P-B', 'P-A', 'C']},
    ],

    # ---- Peer messaging ----------------------------------------------------
    'k_peers_per_day'            : 2,                                   # default

    # ---- Network -----------------------------------------------------------
    'network_type'               : 'stochastic_block',                  # default
    'network_params'             : None,                                # default (legacy flat keys below)
    'p_intra'                    : 0.15,                                # default
    'p_inter'                    : 0.05,                                # default
    'block_sizes'                : None,                                # default (auto from politics_id)
    'diagnostics_timeout_s'      : 30.0,                                # default

    # ---- LLM (research canon: local Qwen3 via mlx_lm.server) --------------
    'llm_model'                  : LOCAL_MODEL,                         # default = 'mlx-community/Qwen3-8B-4bit'
    'llm_provider'               : 'local',                             # default
    'llm_temperature'            : 0.5,                                 # default
    'survey_model'               : None,                                # default (use llm_model for surveys too)
    'survey_provider'            : None,                                # default
    'thinking'                   : True,                                # research canon (Qwen3 thinking on)
    'debias'                     : True,                                # default (Condition B 2-step survey)

    # ---- Communication mode + package policies ----------------------------
    'communication_mode'         : 'package',                           # default
    'package_policies'           : list(ALL_CLIMATE_POLICIES),          # default (ALL six)

    # ---- Political messaging (offline pool = research default) ------------
    'political_message_source'   : 'offline',                           # default
    'political_message_set'      : 'v1',                                # default

    # ---- Day-0 anchoring (research canon) ---------------------------------
    # ground_truth_with_rationale seeds the numeric Day-0 answer from
    # YouGov and the LLM only writes a rationale; assemble_context surfaces
    # that rationale verbatim via the (new, post-NB34) _section_day0_anchor.
    'day0_anchor'                : 'ground_truth_with_rationale',       # default

    # ---- Political-agent reach + audience ---------------------------------
    'reach_a'                    : 1.0,                                 # default
    'reach_b'                    : 1.0,                                 # default
    'audience_cap'               : None,                                # default

    # ---- Political exposure assignment ------------------------------------
    'political_exposure_mode'    : 'rule_affinity_rank',                # default
    'political_exposure_targets' : None,                                # default (committed_minority_symmetric preset)
    'affinity_weights'           : None,                                # default (balanced preset)

    # ---- Reproducibility + IO ---------------------------------------------
    'random_seed'                : RANDOM_SEED,                         # default
    'output_dir'                 : 'data/output/experiments',           # default

    # ---- Local-LLM runtime knobs (provider='local') -----------------------
    'local_base_url'             : LOCAL_BASE_URL,                      # default = None → env CAG_LOCAL_BASE_URL or http://localhost:8080/v1
    'local_extra_body'           : None,                                # default
    'local_timeout_s'            : None,                                # default = None → env CAG_LOCAL_TIMEOUT_S or 600s

    # ---- v0.6 timeline sampling -------------------------------------------
    'timeline_sample_size'       : 3,                                   # default (one agent per top-3 buckets)
    'timeline_sample_agent_ids'  : None,                                # default (auto-stratified)
}

# Sanity check: every SIM_CONFIG key is listed here.
missing = sorted(set(SIM_CONFIG) - set(config))
extra   = sorted(set(config)     - set(SIM_CONFIG))
assert not missing, f'NB34 config is missing keys present in SIM_CONFIG: {missing}'
assert not extra,   f'NB34 config has keys not in SIM_CONFIG: {extra}'
print(f'Config has all {len(config)} SIM_CONFIG keys explicit.')

print()
print(f'n_citizens         = {config["n_citizens"]}  (smoke)')
print(f'days               = {len(config["days"])} alternating package-mode days')
print(f'k_peers_per_day    = {config["k_peers_per_day"]}')
print(f'communication_mode = {config["communication_mode"]}')
print(f'package_policies   = ALL {len(config["package_policies"])} climate policies')
print(f'LLM                = {config["llm_provider"]}/{config["llm_model"]} @ {config["local_base_url"]}')
print(f'debias             = {config["debias"]}, thinking = {config["thinking"]}')
print(f'day0_anchor        = {config["day0_anchor"]}')
print(f'timeline_sample_size = {config["timeline_sample_size"]}  (auto-stratified by bucket)')
print(f'seed               = {config["random_seed"]}')


## 4. Instrument `send_chat` for call counting

Wraps `cag.abm.agent.send_chat` (every LLM call in the simulation flows through this single name) with a counter that buckets each call by category, so after the run we can read off:
- total LLM calls
- calls spent on **Day-0 anchor compression** (NEW in v2)
- calls spent on **daily-summary compression** (count unchanged vs v1, but input is now unified across all policies in package mode)
- everything else (broadcasts, peer messages, reflections, EOD surveys)


In [ ]:
import cag.abm.agent as _agent_mod
from collections import Counter

_orig_send_chat = _agent_mod.send_chat
LLM_CALLS = Counter()

def _counting_send_chat(system_prompt, user_prompt, *args, **kwargs):
    if system_prompt == "You are a concise summariser.":
        if user_prompt.startswith("Concisely summarise your Day-0 position"):
            LLM_CALLS["anchor_compression"] += 1
        elif user_prompt.startswith("Concisely summarise the following day"):
            LLM_CALLS["daily_summary_compression"] += 1
        else:
            LLM_CALLS["other_summariser"] += 1
    else:
        LLM_CALLS["other"] += 1
    LLM_CALLS["total"] += 1
    return _orig_send_chat(system_prompt, user_prompt, *args, **kwargs)

_agent_mod.send_chat = _counting_send_chat
print("send_chat wrapped. LLM_CALLS will be reset by the run cell.")


## 5. Run simulation (timed) + save

Runs the full 4-day package-mode simulation, times wall-clock, then prints:
- elapsed seconds
- per-category LLM call counts (`anchor_compression` should stay at 0 — the step was removed post-NB34; `daily_summary_compression` should equal `n_agents × max(0, n_days - 2)`)


In [ ]:
import time
from cag.abm.sim import run_simulation
from cag.io.results import save_results

LLM_CALLS.clear()
t0 = time.time()
results = run_simulation(config, sn)
wall_s = time.time() - t0
out_dir = save_results(results, output_dir="../data/output/experiments")

print(f"Run dir   : {out_dir}")
print(f"Wall time : {wall_s:.1f} s  ({wall_s/60:.2f} min)")
print()
print("LLM call breakdown:")
for k in ("anchor_compression", "daily_summary_compression", "other_summariser", "other"):
    print(f"  {k:<28} {LLM_CALLS[k]:>5}")
print(f"  {'TOTAL':<28} {LLM_CALLS['total']:>5}")
print()

n_agents  = len(sn.agents_active)
n_days    = len(config["days"])
expected_summary = n_agents * max(0, n_days - 2)

print("Daily-summary call check:")
print(f"  expected = {n_agents} agents x max(0, {n_days}-2) days = {expected_summary}")
print(f"  actual   = {LLM_CALLS['daily_summary_compression']}")
print(f"  anchor_compression bucket: {LLM_CALLS['anchor_compression']} (should be 0 — step removed post-NB34)")


## 7. `manage_memory` runs BEFORE EOD survey (v2 reordering)

In v1 the daily summary for day d-2 was compressed AFTER the day-d EOD survey, so the survey couldn't see it. In v2 `manage_memory` runs above the EOD survey loop, so the survey on day `d` should see `Summary of recent days:` containing a `Day {d-2}:` entry whenever `d >= 3`.

In this 4-day run:
- days 0–2: NO `Summary of recent days:` block in EOD contexts
- day 3: contains `Day 1:`
- day 4: contains `Day 2:` (and possibly `Day 1:`)


In [ ]:
ac = pd.read_csv(Path(out_dir) / "survey_assembled_context.csv")
print(f"survey_assembled_context.csv: {len(ac)} rows across {sorted(ac.day.unique())} days")
print()

checks = []
for d in sorted(ac.day.unique()):
    on_day = ac[ac.day == d]
    if len(on_day) == 0:
        continue
    has_summary = on_day.assembled_context.str.contains("Summary of recent days:", regex=False)
    summary_pct = 100 * has_summary.mean()
    should_have = (d >= 3)
    ok = (summary_pct > 50) == should_have
    checks.append(ok)
    print(f"  day {d}: 'Summary of recent days:' in {summary_pct:5.1f}% of EOD contexts "
          f"(should_have={should_have}) {'OK' if ok else 'FAIL'}")

print()
for d in (3, 4):
    on_day = ac[ac.day == d]
    if len(on_day) == 0:
        continue
    target = d - 2
    marker = f"Day {target}:"
    pct = 100 * on_day.assembled_context.str.contains(marker, regex=False).mean()
    ok = pct > 50
    checks.append(ok)
    print(f"  day {d}: contexts containing '{marker}' (inside summary block) = {pct:5.1f}% {'OK' if ok else 'FAIL'}")

print(f"\nsection 7: {sum(checks)}/{len(checks)} checks passed")
section7_ok = all(checks)


## 8. EOD survey policy-order shuffle (deterministic per-day, identical across agents)

v1 walked policies in fixed `1 → 6` order, baking in a strong position bias (e.g. carbon-tax always answered 5th, after the today-so-far block had already filled with 4 other policy answers). v2 shuffles per-day using `random.Random(seed*1000 + day)` — one shuffle per day, applied identically to every agent.

We verify three things:
1. **Same across agents within a day** — one shuffle is applied to all agents.
2. **Different across days** — no fixed ordering.
3. **Reproducible** — the order on each day matches `random.Random(seed*1000 + day).shuffle(list(package_policies))` exactly.


In [ ]:
import random as _random

reas_all = pd.read_csv(Path(out_dir) / "survey_reasoning.csv")
package_pids_str = [str(p) for p in config["package_policies"]]

checks = []
day_orders = {}
for d in sorted(reas_all.day.unique()):
    if d == 0:
        continue  # Day-0 baseline loop, not the shuffled EOD path
    rows = reas_all[reas_all.day == d]
    per_agent = (
        rows.sort_values(["agent_id", "sim_step"])
            .groupby("agent_id")["policy_id"]
            .apply(lambda s: tuple(s))
    )
    distinct = set(per_agent)
    same_across_agents = len(distinct) == 1
    actual = per_agent.iloc[0]
    day_orders[d] = actual

    expected = list(package_pids_str)
    _random.Random(int(config["random_seed"]) * 1000 + int(d)).shuffle(expected)
    matches_expected = actual == tuple(expected)

    checks.append(same_across_agents)
    checks.append(matches_expected)

    short = " -> ".join(p.split(".")[-1] for p in actual)
    print(f"  day {d}: same-across-agents={same_across_agents}, matches-expected={matches_expected}")
    print(f"          {short}")

cross_day_distinct = len(set(day_orders.values())) == len(day_orders)
checks.append(cross_day_distinct)
print(f"\n  cross-day distinctness: {len(set(day_orders.values()))} distinct orders across "
      f"{len(day_orders)} EOD days {'OK' if cross_day_distinct else 'FAIL'}")

print(f"\nsection 8: {sum(checks)}/{len(checks)} checks passed")
section8_ok = all(checks)


## 9. `Your answers so far in today's survey:` section (within-day, package mode)

In package mode the EOD survey for the N-th policy in a day's shuffled order should expose the previous N-1 policies inside the `Your answers so far in today's survey:` block. The first policy of the day should NOT have this block.


In [ ]:
ac_ord = ac.sort_values(["agent_id", "day", "sim_step"]).reset_index(drop=True)
ac_ord["pos_in_day"] = ac_ord.groupby(["agent_id", "day"]).cumcount()

HEADER = "Your answers so far in today's survey:"
checks = []
for d in sorted(ac_ord.day.unique()):
    if d == 0:
        continue
    day_rows = ac_ord[ac_ord.day == d]
    first_pct = 100 * (
        day_rows[day_rows.pos_in_day == 0]
        .assembled_context.str.contains(HEADER, regex=False).mean()
    )
    later_pct = 100 * (
        day_rows[day_rows.pos_in_day >= 1]
        .assembled_context.str.contains(HEADER, regex=False).mean()
    )
    ok_first = first_pct < 5
    ok_later = later_pct > 95
    checks.append(ok_first)
    checks.append(ok_later)
    print(f"  day {d}: today-so-far in 1st-of-day = {first_pct:5.1f}% (want ~0)  {'OK' if ok_first else 'FAIL'}")
    print(f"          today-so-far in later-of-day = {later_pct:5.1f}% (want ~100) {'OK' if ok_later else 'FAIL'}")

print(f"\nsection 9: {sum(checks)}/{len(checks)} checks passed")
section9_ok = all(checks)


## 10. Full assembled-context dump (one example) + section-order check

Pick a late-in-day context on day 4 — should have every v2 section populated. Verify section headers appear in the spec order:

1. (persona — no header)
2. `Original prior position on "<label>":`
3. `Summary of recent days:`
4. `Recent reflections following received messages:`
5. `Your considered position in recent days:`
6. `Your answers so far in today's survey:`


In [ ]:
late_day = max(ac_ord.day.unique())
example = ac_ord[ac_ord.day == late_day].sort_values("pos_in_day").iloc[-1]
ctx = example["assembled_context"]
print(f"agent_id={example['agent_id']}, day={example['day']}, "
      f"policy={example['policy_id']}, pos_in_day={example['pos_in_day']}")
print("=" * 80)
print(ctx)
print("=" * 80)

headers = [
    "Original prior position on",
    "Summary of recent days:",
    "Recent reflections following received messages:",
    "Your considered position in recent days:",
    "Your answers so far in today's survey:",
]
positions = [(h, ctx.find(h)) for h in headers]
present_positions = [p for _, p in positions if p >= 0]
ordered_ok = present_positions == sorted(present_positions)

print(f"\nheaders present: {len(present_positions)}/{len(headers)}")
for h, p in positions:
    print(f"  pos={p:>5}  {h}")
print(f"\nsection-ordering correct: {ordered_ok}")
section10_ok = ordered_ok and len(present_positions) >= 4


## 11. Unified daily-summary contents (cross-policy reasoning)

In package mode v2 produces ONE `daily_summaries[(d, PACKAGE_SCOPE)]` row per agent per day-being-compressed (no per-policy explosion), built from every reflection on that day plus every same-day `survey_reasoning` entry across all 6 policies.

Verify:
- exactly `n_agents × max(0, n_days - 2)` rows
- all rows tagged `PACKAGE_SCOPE`
- spot-check one summary (qualitative)


In [ ]:
from cag.abm.attributes.opinion import PACKAGE_SCOPE

ds = pd.read_csv(Path(out_dir) / "daily_summaries.csv")
print(f"daily_summaries.csv: {len(ds)} rows  (expected {expected_summary})")
print(f"  by day:    {ds.day.value_counts().sort_index().to_dict()}")
n_pkg_rows = (ds.policy_id == PACKAGE_SCOPE).sum()
print(f"  rows tagged PACKAGE_SCOPE: {n_pkg_rows} / {len(ds)}")
print()

if len(ds):
    ex = ds.iloc[0]
    print(f"Example summary (agent={ex['agent_id']}, day={ex['day']}, policy_id={ex['policy_id']}):")
    print("-" * 60)
    print(ex["summary"])
    print("-" * 60)

print()
print("Summary length distribution (chars):")
print(ds.summary.str.len().describe().round(1).to_string())

section11_ok = (
    len(ds) == expected_summary
    and n_pkg_rows == len(ds)
)
print(f"\nsection 11 OK: {section11_ok}")


## 12. Scoreboard


In [ ]:
sections = {
    "7. manage_memory before EOD":     section7_ok,
    "8. EOD shuffle determinism":      section8_ok,
    "9. today-so-far section":         section9_ok,
    "10. section ordering":            section10_ok,
    "11. unified daily-summary":       section11_ok,
}
for name, ok in sections.items():
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")

n_pass = sum(sections.values())
print(f"\n{n_pass}/{len(sections)} sections passed")
print()
print(f"Run dir   : {out_dir}")
print(f"Wall time : {wall_s:.1f} s")
print(f"Total LLM : {LLM_CALLS['total']}")
